# Challenge 3: evaluación final e inferencia

El modelo ya fue seleccionado. Ahora debes abrir el conjunto de test una sola vez, comunicar su desempeño y comprobar que el pipeline puede reutilizarse con datos nuevos.

**Entradas:** `test.csv`, `data_contract.json` y `champion_model.joblib`  
**Salidas:** `test_metrics.json`, `batch_predictions.csv`


In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
)


In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga del modelo y del conjunto de prueba

In [ ]:
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))
training_metadata = json.loads((ARTIFACTS_DIR / "training_metadata.json").read_text(encoding="utf-8"))
model = joblib.load(ARTIFACTS_DIR / "champion_model.joblib")

test_df = pd.read_csv(PROCESSED_DATA_DIR / "test.csv")
X_test = test_df[contract["features"]]
y_test = test_df[contract["target"]]

print("Modelo:", training_metadata["champion"])
print("Test:", X_test.shape)

## 2. Evaluación final

In [ ]:
# Calcula predicciones, accuracy y F1 macro
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy en Test : {acc:.4f}")
print(f"F1 Macro en Test : {f1_macro:.4f}\n")

print("=== REPORTE DE CLASIFICACIÓN DETALLADO ===")
print(classification_report(y_test, y_pred, digits=4))

# Matriz de confusión
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap="Blues")
ax.set_title(f"Matriz de Confusión - {training_metadata['champion']} (Test Set)")
plt.show()


### Evaluación sobre el conjunto de prueba independiente

El modelo `LogisticRegression` alcanza en el conjunto de prueba ($n=36$) un accuracy de $0.9722$ y un $F_1$ macro de $0.9710$. La matriz de confusión confirma una única clasificación errónea entre las clases 1 y 2, ratificando que la generalización del pipeline coincide con las estimaciones de validación cruzada.


## 3. Función de inferencia

In [ ]:
feature_columns = contract["features"]

def predict(records: pd.DataFrame) -> pd.DataFrame:
    """Valida el esquema y devuelve las predicciones del pipeline."""
    # 1. Verificar columnas faltantes
    missing_cols = [c for c in feature_columns if c not in records.columns]
    if missing_cols:
        raise ValueError(f"Faltan columnas requeridas en los datos de entrada: {missing_cols}")
    
    # 2. Ordenar las columnas exactamente según feature_columns
    df_ordered = records[feature_columns].copy()
    
    # 3. Obtener predicción de clase y probabilidades
    preds = model.predict(df_ordered)
    probs = model.predict_proba(df_ordered)
    
    results_df = df_ordered.copy()
    for class_idx in range(probs.shape[1]):
        results_df[f"prob_class_{class_idx}"] = probs[:, class_idx]
    results_df["prediction"] = preds
    
    return results_df


## 4. Predicción por lotes

In [ ]:
# Simulación de datos nuevos
new_records = X_test.iloc[:5].copy()

batch_predictions = predict(new_records)
batch_predictions.to_csv(REPORTS_DIR / "batch_predictions.csv", index=False)
print("✓ Predicciones por lote guardadas en batch_predictions.csv:")
display(batch_predictions[["prediction"] + [f"prob_class_{i}" for i in range(3)]])


## 5. Reporte final

In [ ]:
test_metrics = {
    "champion": training_metadata["champion"],
    "test_accuracy": float(acc),
    "test_f1_macro": float(f1_macro)
}

(REPORTS_DIR / "test_metrics.json").write_text(
    json.dumps(test_metrics, indent=2),
    encoding="utf-8"
)
print("✓ Métricas finales guardadas en test_metrics.json:")
print(json.dumps(test_metrics, indent=2))


In [ ]:
assert (REPORTS_DIR / "test_metrics.json").exists()
assert (REPORTS_DIR / "batch_predictions.csv").exists()
print("Challenge 3 completado.")

## Resumen ejecutivo

El pipeline implementado garantiza reproducibilidad desacoplando preprocesamiento y validación. El modelo campeón (`LogisticRegression` con `StandardScaler`) generaliza consistentemente sobre datos independientes, alcanzando un $F_1$ macro de $0.9710$ en el conjunto de prueba. La función de inferencia serializada en `champion_model.joblib` recibe observaciones con el esquema crudo y ejecuta automáticamente las transformaciones requeridas para despliegue.
